In [ ]:
# Load libraries used in this notebook.

import os
import warnings

import dill
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from plotnine import element_text, labs, theme
from scenicplus.cistromes import TF_cistrome_correlation, generate_pseudobulks
from scenicplus.dimensionality_reduction import plot_metadata_given_ax
from scenicplus.plotting.dotplot import heatmap_dotplot

warnings.filterwarnings("ignore")

# Directory containing the SCENIC+ result objects used for PVH visualization.
work_dir = "/path/to/PVH_scripts_scenicplus/"

In [ ]:
# Load motif enrichment results and the SCENIC+ object with eRegulon scores.
with open(os.path.join(work_dir, "motif_enrichment_results", "menr.pkl"), "rb") as f:
    menr = dill.load(f)



In [ ]:
# Inspect one example motif-enrichment topic.

print("Plot topics example from menr object: Topic 3")
menr["DEM_Topics_All"].DEM_results("Topic3")

In [ ]:
# Do this part in a High Performance Computing Cluster; it is too intensive to run locally.

import warnings
import os
warnings.simplefilter(action='ignore', category=FutureWarning)
import sys
_stderr = sys.stderr
null = open(os.devnull, 'wb')

work_dir = '/path/to/PVH_SCENIC/'

import pycistarget
import dill
import scenicplus

scplus_obj = dill.load(open(os.path.join(work_dir, 'scenicplus/scplus_obj.pkl'), 'rb'))

from scenicplus.preprocessing.filtering import apply_std_filtering_to_eRegulons
apply_std_filtering_to_eRegulons(scplus_obj)

scplus_obj.uns['eRegulon_metadata_filtered'].head()

# Add split_clusters.
scplus_obj.metadata_cell['split_clusters'] = scplus_obj.metadata_cell['GEX_cluster'].astype(str) + '_' + scplus_obj.metadata_cell['ACC_group']
scplus_obj.metadata_cell['split_clusters'].unique()

import numpy as np

scplus_obj.metadata_cell['split_clusters'] = scplus_obj.metadata_cell['split_clusters'].astype(str)

from scenicplus.eregulon_enrichment import score_eRegulons
region_ranking = dill.load(open(os.path.join(work_dir, 'scenicplus/region_ranking.pkl'), 'rb'))  # Load ranking calculated using the wrapper function.
gene_ranking = dill.load(open(os.path.join(work_dir, 'scenicplus/gene_ranking.pkl'), 'rb'))  # Load ranking calculated using the wrapper function.

score_eRegulons(
    scplus_obj,
    ranking=region_ranking,
    eRegulon_signatures_key='eRegulon_signatures_filtered',
    key_added='eRegulon_AUC_filtered',
    enrichment_type='region',
    auc_threshold=0.05,
    normalize=False,
    n_cpu=5
)

score_eRegulons(
    scplus_obj,
    gene_ranking,
    eRegulon_signatures_key='eRegulon_signatures_filtered',
    key_added='eRegulon_AUC_filtered',
    enrichment_type='gene',
    auc_threshold=0.05,
    normalize=False,
    n_cpu=5
)

from scenicplus.dimensionality_reduction import run_eRegulons_tsne, run_eRegulons_umap
run_eRegulons_umap(
    scplus_obj=scplus_obj,
    auc_key='eRegulon_AUC_filtered',
    reduction_name='eRegulons_UMAP'
)
run_eRegulons_tsne(
    scplus_obj=scplus_obj,
    auc_key='eRegulon_AUC_filtered',
    reduction_name='eRegulons_tSNE'
)

dill.dump(scplus_obj, open(os.path.join(work_dir, 'scenicplus/split_clusters_updated_scplus_obj.pkl'), 'wb'), protocol=-1)

In [ ]:
# Ensure cluster labels are strings and define the cluster-condition grouping used for heatmaps.

# Open HPC-generated file.
with open(os.path.join(work_dir, "scenicplus", "split_clusters_updated_scplus_obj.pkl"), "rb") as f:
    scplus_obj = dill.load(f)

scplus_obj.metadata_cell["GEX_cluster"] = scplus_obj.metadata_cell["GEX_cluster"].astype(str)
scplus_obj.metadata_cell["split_clusters"] = (
    scplus_obj.metadata_cell["ACC_cluster"].astype(str) + "_" +
    scplus_obj.metadata_cell["ACC_group"].astype(str)
)

In [ ]:
# Plot eRegulon embeddings colored by RNA cluster annotation.

print("Plot cluster per condition")

%matplotlib inline

fig, axs = plt.subplots(ncols=2, figsize=(16, 8))
plot_metadata_given_ax(
    scplus_obj=scplus_obj,
    ax=axs[0],
    reduction_name="eRegulons_UMAP",
    variable="GEX_cluster"
)
plot_metadata_given_ax(
    scplus_obj=scplus_obj,
    ax=axs[1],
    reduction_name="eRegulons_tSNE",
    variable="GEX_cluster"
)
fig.tight_layout()
sns.despine(ax=axs[0])
sns.despine(ax=axs[1])
plt.show()

In [ ]:
# Correlate TF expression with region-based cistrome activity using RNA clusters as pseudobulk groups.

generate_pseudobulks(
    scplus_obj=scplus_obj,
    variable="GEX_cluster",
    auc_key="eRegulon_AUC_filtered",
    signature_key="Region_based"
)

TF_cistrome_correlation(
    scplus_obj,
    use_pseudobulk=True,
    variable="GEX_cluster",
    auc_key="eRegulon_AUC_filtered",
    signature_key="Region_based",
    out_key="filtered_region_based"
)

In [ ]:
# Display TF-cistrome correlations ranked by adjusted p-value.

print("Transcription factor correlation to regulatory regions")
scplus_obj.uns["TF_cistrome_correlation"]["filtered_region_based"].sort_values(
    by="Adjusted_p-value"
)

In [ ]:
# Plot correlation strength and significance for region-based cistromes.

print("Plot enrichment significance for eRegulons")

n_targets = [
    int(x.split("(")[1].replace("r)", ""))
    for x in scplus_obj.uns["TF_cistrome_correlation"]["filtered_region_based"]["Cistrome"]
]
rho = scplus_obj.uns["TF_cistrome_correlation"]["filtered_region_based"]["Rho"].to_list()
adj_pval = scplus_obj.uns["TF_cistrome_correlation"]["filtered_region_based"]["Adjusted_p-value"].to_list()

thresholds = {
    "rho": [-0.75, 0.70],
    "n_targets": 0
}

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(rho, n_targets, c=-np.log10(adj_pval), s=5)
ax.set_xlabel("Correlation coefficient")
ax.set_ylabel("nr. target regions")
ax.vlines(x=thresholds["rho"], ymin=0, ymax=max(n_targets), color="black", ls="dashed", lw=1)
ax.text(x=thresholds["rho"][0], y=max(n_targets), s=str(thresholds["rho"][0]))
ax.text(x=thresholds["rho"][1], y=max(n_targets), s=str(thresholds["rho"][1]))
sns.despine(ax=ax)
fig.colorbar(sc, label="-log10(adjusted_pvalue)", ax=ax)
plt.show()

In [ ]:
# Manually define the cluster-condition order used in the eRegulon heatmaps.

selected_indexes_sorted = [
    "3.0_OB", "3.0_Leptin3hr", "3.0_Leptin24hr", "3.0_Leptin72hr", "3.0_WT",
    "6.0_OB", "6.0_Leptin3hr", "6.0_Leptin24hr", "6.0_Leptin72hr", "6.0_WT",
    "7.0_OB", "7.0_Leptin3hr", "7.0_Leptin24hr", "7.0_Leptin72hr", "7.0_WT",
    "8.0_OB", "8.0_Leptin3hr", "8.0_Leptin24hr", "8.0_Leptin72hr", "8.0_WT",
    "11.0_OB", "11.0_Leptin3hr", "11.0_Leptin24hr", "11.0_Leptin72hr", "11.0_WT",
    "14.0_OB", "14.0_Leptin3hr", "14.0_Leptin24hr", "14.0_Leptin72hr", "14.0_WT",
    "15.0_OB", "15.0_Leptin3hr", "15.0_Leptin24hr", "15.0_Leptin72hr", "15.0_WT",
    "16.0_OB", "16.0_Leptin3hr", "16.0_Leptin24hr", "16.0_Leptin72hr", "16.0_WT",
    "18.0_OB", "18.0_Leptin3hr", "18.0_Leptin24hr", "18.0_Leptin72hr", "18.0_WT",
    "22.0_OB", "22.0_Leptin3hr", "22.0_Leptin24hr", "22.0_Leptin72hr", "22.0_WT"
]

valid_index = scplus_obj.metadata_cell["split_clusters"].unique()
filtered_index_order = [label for label in selected_indexes_sorted if label in valid_index]

print(filtered_index_order)

In [ ]:
# Plot all region-based eRegulons across cluster-condition groups.

print("Plot enrichment of eRegulons per cluster per condition")

heatmap_dotplot(
    scplus_obj=scplus_obj,
    size_matrix=scplus_obj.uns["eRegulon_AUC_filtered"]["Region_based"],
    color_matrix=scplus_obj.to_df("EXP"),
    scale_size_matrix=True,
    scale_color_matrix=True,
    group_variable="split_clusters",
    index_order=filtered_index_order,
    figsize=(20, 20),
    orientation="vertical"
)

In [ ]:
# Plot activator eRegulons only and save the heatmap to PDF.

auc = scplus_obj.uns["eRegulon_AUC_filtered"]["Region_based"]
ereg_activators = [c for c in auc.columns if ("_+_(" in c) and ("_-_(" not in c)]
auc_act = auc.loc[:, ereg_activators].copy()

p = heatmap_dotplot(
    scplus_obj=scplus_obj,
    size_matrix=auc_act,
    color_matrix=scplus_obj.to_df("EXP"),
    scale_size_matrix=True,
    scale_color_matrix=True,
    group_variable="split_clusters",
    index_order=filtered_index_order,
    figsize=(10, 10),
    orientation="vertical",
    split_repressor_activator=False,
    save=None
)

p = p + theme(
    axis_text_x=element_text(
        rotation=90,
        ha="center",
        va="top",
        size=10
    )
)

p = p + labs(size="Scaled region-based eRegulon AUC")
p.scales.get_scales("fill").name = "Scaled TF expression"

p.save(
    "/path/to/PVH_activators_eRegulon_heatmap.pdf",
    width=20,
    height=20,
    units="in"
)